# Frozen 3 - real ColQwen index + query cache (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BiprashPandey/altVVRAG/blob/main/notebooks/frozen3_colab.ipynb)

Run top to bottom. **No uploads and no local files are required.** The notebook:

1. installs the GPU stack;
2. clones the exact `main` commit (code + frozen inputs);
3. verifies the frozen gold/probes/deck-list hashes;
4. materializes the **exact 25 frozen decks** from the SlideVQA mirror;
5. builds the real ColQwen index;
6. encodes all **180** frozen questions (120 gold + 60 probes);
7. validates retrieval and probe absence;
8. packages `frozen3_gpu_bundle.zip` and downloads it.

**Before running:** Runtime -> Change runtime type -> T4 GPU. Do **not** edit
labels, questions, or the frozen JSON files.

## 0. GPU check

A T4 is enough. If CUDA is false, stop and change the runtime type.

In [ ]:
!nvidia-smi || echo 'NO GPU VISIBLE -> Runtime > Change runtime type > T4 GPU'
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'CUDA is required for the real ColQwen index'

## 1. Install the GPU stack

Colab already provides a CUDA torch build. `colpali-engine` supplies ColQwen2;
Tesseract provides the OCR text layer.

In [ ]:
%pip install -q colpali-engine qdrant-client pymupdf pillow pytesseract transformers datasets pyarrow groq google-genai
%pip install -q -U torchao
!apt-get install -y tesseract-ocr > /dev/null 2>&1
!tesseract --version | head -1

## 2. Clone the exact `main` commit

The frozen inputs are tracked in the repo (JSON only; the 1.1 GB raw decks are
downloaded later). If `/content/altVRAG` already exists, reset only tracked
files so a resumed runtime keeps `data/index_frozen3` and the query cache.

In [ ]:
%cd /content
!if [ -d altVRAG/.git ]; then git -C altVRAG fetch --depth 1 origin main && git -C altVRAG reset --hard origin/main; else git clone --depth 1 https://github.com/BiprashPandey/altVVRAG.git altVRAG; fi
%cd /content/altVRAG
!git log -1 --oneline
!pip install -q -e .

## 3. Verify the frozen inputs

This fails early if the clone is stale, a JSON file was edited, or the deck list
does not match the frozen metadata.

In [ ]:
import hashlib, json
from pathlib import Path

REPO = Path('/content/altVRAG')
SETD = REPO / 'data' / 'slidevqa_frozen3'
META = json.loads((SETD / 'frozen_meta.json').read_text())

for name, key in [
    ('gold.json', 'gold_sha256'),
    ('probes.json', 'probes_sha256'),
    ('deck_ids.json', 'deck_ids_sha256'),
    ('questions_all.json', 'questions_all_sha256'),
]:
    got = hashlib.sha256((SETD / name).read_bytes()).hexdigest()
    assert got == META[key], f'{name} hash mismatch: {got} != {META[key]}'

gold = json.loads((SETD / 'gold.json').read_text())
probes = json.loads((SETD / 'probes.json').read_text())
decks = json.loads((SETD / 'deck_ids.json').read_text())
questions = json.loads((SETD / 'questions_all.json').read_text())
assert len(gold) == 120, f'gold count {len(gold)} != 120'
assert len(probes) == 60, f'probe count {len(probes)} != 60'
assert len(decks) == 25, f'deck count {len(decks)} != 25'
assert len(questions) == 180, f'question count {len(questions)} != 180'
assert len({x['qid'] for x in gold + probes}) == 180, 'duplicate qids'
assert (REPO / 'config.frozen3.text.yaml').exists(), 'missing frozen3 config'

print('frozen inputs OK:',
      f'{len(decks)} decks | {len(gold)} gold | {len(probes)} probes')
print('frozen_meta:', META.get('created_utc'), '| audit:', META.get('audit'))

## 4. Materialize the exact frozen decks

The mirror is streamed; only the 25 deck IDs in `deck_ids.json` are written.
The frozen `gold.json` is never regenerated. This step is skipped when the 25
PDFs are already present.

In [ ]:
import os, subprocess, sys
os.chdir(REPO)
raw = SETD / 'raw'
pdfs = sorted(raw.glob('*.pdf')) if raw.exists() else []
if len(pdfs) != len(decks):
    subprocess.run([
        sys.executable, 'scripts/prepare_slidevqa.py',
        '--out', str(SETD), '--split', 'train',
        '--decks-file', str(SETD / 'deck_ids.json'),
        '--gold-out', 'gold_rebuilt.json',
        '--max-rows', '20000', '--n-decks', '9999', '--n-qa', '0',
    ], check=True)
else:
    print(f'{len(pdfs)} frozen PDFs already present; skipping download')

In [ ]:
import pymupdf
pdfs = sorted((SETD / 'raw').glob('*.pdf'))
pages = 0
for p in pdfs:
    with pymupdf.open(p) as d:
        pages += d.page_count
print('decks wanted', len(decks), '| pdfs', len(pdfs), '| pages', pages)
assert len(pdfs) == len(decks) == 25, 'deck count mismatch'
assert pages == 500, f'page count mismatch: {pages}'
print('materialization OK')

## 5. Build the real ColQwen index

This is the slow step: Tesseract OCR + ColQwen2 page embeddings at 150 dpi with
2x OCR upscaling. It resumes if `/content/altVRAG/data/index_frozen3` survives.

In [ ]:
index = REPO / 'data' / 'index_frozen3'
if not (index / 'manifest.json').exists():
    subprocess.run([
        sys.executable, '-m', 'vvrag.ingest.run_ingest',
        '--input', 'data/slidevqa_frozen3/raw',
        '--out', 'data/index_frozen3',
        '--embedder', 'colqwen',
        '--model-id', 'vidore/colqwen2-v1.0',
        '--device', 'cuda', '--dtype', 'bfloat16',
        '--dpi', '150', '--upscale', '2.0',
        '--qdrant-path', 'data/index_frozen3/qdrant',
        '--batch-size', '4',
    ], check=True)
else:
    print('index manifest already present; skipping ingest')

In [ ]:
import sqlite3
manifest = json.loads((index / 'manifest.json').read_text())
con = sqlite3.connect(index / 'meta.db')
pages_n = con.execute('select count(*) from pages').fetchone()[0]
words_n = con.execute('select count(*) from word_boxes').fetchone()[0]
sent_n = con.execute('select count(*) from sentences').fetchone()[0]
con.close()
print(json.dumps(manifest, indent=2))
print('pages', pages_n, '| word_boxes', words_n, '| sentences', sent_n)
assert pages_n == 500, f'index pages {pages_n} != 500'
assert words_n > 0 and sent_n > 0, 'empty OCR/sentence index'
print('index OK')

## 6. Encode all 180 frozen questions

The cache key is `sha1(question)`; hybrid retrieval reads
`data/query_cache_frozen3` on the laptop. The second command verifies coverage.

In [ ]:
subprocess.run([
    sys.executable, 'scripts/precompute_query_vectors.py',
    '--gold', 'data/slidevqa_frozen3/questions_all.json',
    '--out', 'data/query_cache_frozen3',
    '--model', 'vidore/colqwen2-v1.0',
    '--device', 'cuda',
], check=True)
subprocess.run([
    sys.executable, 'scripts/precompute_query_vectors.py',
    '--gold', 'data/slidevqa_frozen3/questions_all.json',
    '--out', 'data/query_cache_frozen3',
    '--check',
], check=True)
print('query cache OK')

## 7. Validate retrieval and probe absence before any live model call

Keyword vs visual vs hybrid page-hit/MRR is written to
`results/frozen3_retrieval_validation/rows.json`. Every probe's `check_absent`
string must be absent from the new real OCR.

In [ ]:
Path('results').mkdir(exist_ok=True)
subprocess.run([
    sys.executable, 'scripts/validate_retrieval.py',
    '--index', 'data/index_frozen3',
    '--gold', 'data/slidevqa_frozen3/gold.json',
    '--query-cache', 'data/query_cache_frozen3',
    '--out', 'results/frozen3_retrieval_validation',
], check=True)
print((REPO / 'results' / 'frozen3_retrieval_validation' / 'rows.json').read_text()[:2000])

In [ ]:
from vvrag.reader.claims import normalize_answer

con = sqlite3.connect(index / 'meta.db')
text = ' '.join(r[0] or '' for r in con.execute('select text from word_boxes'))
text += ' ' + ' '.join(r[0] or '' for r in con.execute('select text from sentences'))
con.close()
norm = normalize_answer(text)
bad = []
for p in probes:
    for s in p.get('check_absent', []):
        if normalize_answer(s) and normalize_answer(s) in norm:
            bad.append((p['qid'], s))
print('probes', len(probes), '| present checks', len(bad))
assert not bad, bad[:5]
print('probe absence OK on real OCR')

## 8. Record Colab provenance

This JSON is bundled with the index so every later result can be tied to the
exact commit, GPU, model, index counts, and query-cache coverage.

In [ ]:
import datetime
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
cache_files = sorted((REPO / 'data' / 'query_cache_frozen3').glob('*.json'))
prov = {
    'created_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'repo': 'https://github.com/BiprashPandey/altVVRAG',
    'commit': commit,
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
    'colqwen_model': 'vidore/colqwen2-v1.0',
    'index_manifest': manifest,
    'index_pages': pages_n,
    'index_word_boxes': words_n,
    'index_sentences': sent_n,
    'n_frozen_decks': len(pdfs),
    'n_frozen_pages': pages,
    'n_gold': len(gold),
    'n_probes': len(probes),
    'n_questions': len(questions),
    'n_query_vectors': len(cache_files),
}
Path('results/colab_frozen3_provenance.json').write_text(json.dumps(prov, indent=2))
print(json.dumps(prov, indent=2))

## 9. Package and download

The bundle contains the real index, query cache, retrieval validation,
provenance, and frozen JSON files. Send it back for the laptop live run; do not
edit any frozen file afterward.

In [ ]:
!rm -f /content/frozen3_gpu_bundle.zip
!zip -qr /content/frozen3_gpu_bundle.zip data/index_frozen3 data/query_cache_frozen3 results/frozen3_retrieval_validation results/colab_frozen3_provenance.json data/slidevqa_frozen3/frozen_meta.json data/slidevqa_frozen3/gold.json data/slidevqa_frozen3/probes.json data/slidevqa_frozen3/deck_ids.json data/slidevqa_frozen3/questions_all.json
!ls -lh /content/frozen3_gpu_bundle.zip
from google.colab import files
files.download('/content/frozen3_gpu_bundle.zip')

## If an assertion fails

Stop and report the failing cell verbatim. Do not patch the frozen JSON files,
regenerate labels, or change the question list. A failed hash/probe check means
the set is not frozen and the run must not be treated as a frozen3 result.